 ## 1: Data Preparation & Evaluation Dataset Creation:


In [36]:
import os
import json
import re
import time
import random
import shutil
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import Dataset
from peft import (
    PeftModel,
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document

random.seed(42)

print(f"CUDA available: {torch.cuda.is_available()}")



CUDA available: True


 **1.1: Load and Explore the data (course_full.json)**

In [15]:
#Load data
with open("course_full.json", 'r') as f:
    courses = json.load(f)
print(f"Total courses loaded: {len(courses)}")

#Explore the structure
print("exporing data structure:")


#first course data:
sample_course = courses[0]
print(f"\nSample course: {sample_course['courseCode']} - {sample_course['name']}")
print(f"\nAvailable fields: {list(sample_course.keys())}")

#fields
text_fields = ['name', 'nameAlt', 'purpose', 'goal', 'content', 'prerequisites']
print(f"\n\nText content lengths:")
for field in text_fields:
    if sample_course.get(field):
        length = len(str(sample_course[field]))
        print(f"  {field}: {length} chars")
        
#Show sample content with HTML
print(f"\n\nSample 'purpose' field (with HTML):")
print(sample_course['purpose'][:300])

print(f"\n\nSample 'goal' field (with HTML):")
print(sample_course['goal'][:300])

Total courses loaded: 1388
exporing data structure:

Sample course: FSP047 - English for engineers

Available fields: ['courseCode', 'acYear', 'id', 'discont', 'newChanges', 'name', 'nameAlt', 'adoptedDate', 'owner', 'ownerId', 'credit', 'creditNumeric', 'unit', 'eduLevel', 'eduLevelName', 'grading', 'gradingName', 'dept', 'deptBenamning', 'mtsS', 'mjoS', 'adpS', 'unitName', 'unitShort', 'mtsCredit', 'mjoCredit', 'adpCredit', 'rounds', 'genEntryReq', 'specEntryReq', 'purpose', 'goal', 'prerequisites', 'content', 'organization', 'litterature', 'examination', 'updated', 'isLastCurrentForAcYear', 'mainSubjects', 'changes', 'themes', 'sctwGoals', 'courseRounds', 'replaces', 'suggest', 'AI_summary']


Text content lengths:
  name: 21 chars
  nameAlt: 31 chars
  purpose: 248 chars
  goal: 878 chars
  content: 918 chars
  prerequisites: 363 chars


Sample 'purpose' field (with HTML):
<p>The aim of the course is to improve students level of proficiency in English in different communicative si

**1.2: Clean/preprocess course data (remove HTML tags, format text):**

In [16]:
# function to remove HTML tags and clean text
def clean_html_text(html_str):
    if not html_str:
        return ""
    soup = BeautifulSoup(html_str, "html.parser")
    text = soup.get_text().replace("\u0092", "'")
    # Clean up whitespace and replace multiple spaces/newlines with single space and then strip
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# function to clean all courses in the dataset
def clean_course_data(courses):
    cleaned_courses = []
    for course in courses:
        cleaned = {
            'courseCode': course['courseCode'],
            'name': course['name'],
            'nameAlt': course['nameAlt'],
            'creditNumeric': course['creditNumeric'],
            'eduLevelName': course['eduLevelName'],
            'deptBenamning': course['deptBenamning'],
            
            'purpose': clean_html_text(course['purpose']),
            'goal': clean_html_text(course['goal']),
            'content': clean_html_text(course['content']),
            'prerequisites': clean_html_text(course['prerequisites']),
            'examination': clean_html_text(course['examination']),
            
            'AI_summary': course.get('AI_summary', '')
        }
        
        cleaned['full_text'] = (
            f"Course level: {cleaned['eduLevelName']}. " +
            f"Department: {cleaned['deptBenamning']}. " +
            f"Credits: {cleaned['creditNumeric']} HP. " +
            f"Course name: {cleaned['name']}. " +
            cleaned['purpose'] + " " +
            cleaned['goal'] + " " +
            cleaned['content'] + " " +
            cleaned['prerequisites'] + " " +
            cleaned['examination']
        )
        
        cleaned_courses.append(cleaned)
    return cleaned_courses
    
cleaned = clean_course_data(courses)

#the results:
print(f"Cleaned {len(cleaned)} courses")
print(f"\nSample course: {cleaned[0]['courseCode']}")
print(f"  Name: {cleaned[0]['name']}")
print(f"  Credits: {cleaned[0]['creditNumeric']}")
print(f"  Level: {cleaned[0]['eduLevelName']}")
print(f"  Full text length: {len(cleaned[0]['full_text'])} chars")
print(f"\nFull text for the first course:")
print(cleaned[0]['full_text'][:])
#Checking a few more courses:
for i in [0, 50, 100, 300, 400, 500]:
    print(f"\n{cleaned[i]['courseCode']}: {cleaned[i]['name']}")
    print(f"  Full text: {len(cleaned[i]['full_text'])} chars")
    print(f"  Has AI summary: {'Yes' if cleaned[i]['AI_summary'] else 'No'}")
    print(f"  Department: {cleaned[i]['deptBenamning']}")

# Save
with open('cleaned_courses.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned, f, ensure_ascii=False, indent=2)



Cleaned 1388 courses

Sample course: FSP047
  Name: English for engineers
  Credits: 7.5
  Level: First-cycle
  Full text length: 2349 chars

Full text for the first course:
Course level: First-cycle. Department: COMMUNICATION AND LEARNING IN SCIENCE. Credits: 7.5 HP. Course name: English for engineers. The aim of the course is to improve students' level of proficiency in English in different communicative situations. This means developing students' ability in English for academic studies at advanced level as well as in professional life. The successful student is able to use English effectively and appropriately in advanced academic studies as well as in professional settings. This includes the following:write and revise a variety of texts, both academic and professional.select and structure information in paragraphs, sections, and chapters, keeping the audience in mind.adapt and present information for different oral communication contexts and different audiences.analyse sentences fr

 **1.3: Create evaluation dataset:**
210 queries across 6 types:
Type 1: topic_search (35 queries)
Type 2: level_topic (35 queries)  
Type 3: credit_topic (35 queries)
Type 4: specific_lookup (35 queries)
Type 5: swedish_topic_search (35 queries)
Type 6: prerequisite_lookup (35 queries) ← NEW!
Balanced dataset (35 each) 
Ground truth sizes vary (1 for specific lookup, 5+ for topic searches)



In [17]:
# Generic teaching phrases (topic-agnostic)
TEACH_REGEXES = [
    r"introduction to\s+{topic}",
    r"introduces\s+.*{topic}",
    r"fundamental principles of\s+{topic}",
    r"basic principles of\s+{topic}",
    r"course on\s+{topic}",
    r"covers\s+.*{topic}",
    r"{topic}\s+theory",
    r"{topic}\s+fundamentals",
    r"learn\s+{topic}",
    r"learning\s+{topic}",
    r"teaches\s+{topic}",
    r"advanced\s+.*{topic}",
    r"{topic}\s+principles",
    r"{topic}\s+methods",
    r"applied\s+{topic}",
    r"{topic}\s+techniques"
]

def normalize(text):
    return re.sub(r"\s+", " ", text.lower())

# General teaching detector
def teaches_topic(course, topic):
    topic = topic.lower()
    name = normalize(course["name"])
    text = normalize(course["full_text"])

    if name == topic:
        return True
    if topic in name:
        return True

    for pattern in TEACH_REGEXES:
        if re.search(pattern.format(topic=re.escape(topic)), text):
            return True
    return False

def extract_prerequisite_codes(prerequisites_text):
    if not prerequisites_text:
        return []
    pattern = r'\b[A-Z]{3,6}\d{3}\b'
    return list(set(re.findall(pattern, prerequisites_text)))

# Index builder

def build_course_indexes(cleaned_courses, topics, swedish_topics):
    topic_index = {}
    level_index = {}
    credit_index = {}
    swedish_index = {}
    prerequisite_index = {}

    for course in cleaned_courses:
        code = course["courseCode"]
        full_text = normalize(course["full_text"])
        sw_name = normalize(course["nameAlt"])

        level_index.setdefault(course["eduLevelName"], []).append(code)
        credit_index.setdefault(course["creditNumeric"], []).append(code)

        # word-boundary topic matching
        for topic in topics:
            if re.search(rf"\b{re.escape(topic)}\b", full_text):
                topic_index.setdefault(topic, {"teaches": [], "uses": []})
                if teaches_topic(course, topic):
                    topic_index[topic]["teaches"].append(code)
                else:
                    topic_index[topic]["uses"].append(code)

        for s_topic in swedish_topics:
            if s_topic in sw_name:
                swedish_index.setdefault(s_topic, []).append(code)

        prereq_codes = extract_prerequisite_codes(course["prerequisites"])
        if prereq_codes:
            prerequisite_index[code] = prereq_codes

    return {
        "topic": topic_index,
        "level": level_index,
        "credit": credit_index,
        "swedish": swedish_index,
        "prerequisite": prerequisite_index
    }

# Query generators
def create_topic_queries(indexes, topics, target=35):
    queries = []
    templates = [
        "Find courses about {topic}",
        "What courses teach {topic}?",
        "I want to learn about {topic}",
        "Show me {topic} courses"
    ]

    per_topic = target // len(topics) + 1

    for topic in topics:
        courses = indexes["topic"].get(topic, {}).get("teaches", [])
        if not courses:
            continue

        for i in range(per_topic):
            if len(queries) >= target:
                break
            queries.append({
                "query": templates[i % len(templates)].format(topic=topic),
                "ground_truth": courses,
                "query_type": "topic_search",
                "language": "en"
            })

    return queries[:target]

def create_level_topic_queries(indexes, topics, levels, target=35):
    queries = []
    for level in levels:
        for topic in topics:
            teaches = indexes["topic"].get(topic, {}).get("teaches", [])
            matching = list(set(indexes["level"].get(level, [])) & set(teaches))
            if matching:
                queries.append({
                    "query": f"{level} courses about {topic}",
                    "ground_truth": matching,
                    "query_type": "level_topic",
                    "language": "en"
                })

    while len(queries) < target:
        queries.append(random.choice(queries))
    return queries[:target]

def create_credit_topic_queries(indexes, topics, target=35):
    queries = []
    credits = [7.5, 15.0, 30.0, 60.0]

    for credit in credits:
        for topic in topics:
            teaches = indexes["topic"].get(topic, {}).get("teaches", [])
            matching = list(set(indexes["credit"].get(credit, [])) & set(teaches))
            if matching:
                queries.append({
                    "query": f"{credit} credit courses about {topic}",
                    "ground_truth": matching,
                    "query_type": "credit_topic",
                    "language": "en"
                })

    while len(queries) < target:
        queries.append(random.choice(queries).copy())
    return queries[:target]

def create_specific_course_queries(cleaned_courses, target=35):
    queries = []
    selected = random.sample(cleaned_courses, min(len(cleaned_courses), (target + 3) // 4))
    templates = [
        "What is {code}?",
        "Tell me about {code}",
        "Describe {code}",
        "Give me information about {code}"
    ]

    for c in selected:
        for t in templates:
            if len(queries) >= target:
                break
            queries.append({
                "query": t.format(code=c["courseCode"]),
                "ground_truth": [c["courseCode"]],
                "query_type": "specific_lookup",
                "language": "en"
            })
    return queries[:target]

def create_swedish_topic_queries(swedish_index, swedish_topics, target=35):
    queries = []
    templates = [
        "Hitta kurser om {topic}",
        "Vilka kurser handlar om {topic}?",
        "Jag vill lära mig om {topic}",
        "Visa mig kurser om {topic}"
    ]

    per_topic = target // len(swedish_topics) + 1
    for topic in swedish_topics:
        courses = swedish_index.get(topic, [])
        if not courses:
            continue
        for i in range(per_topic):
            if len(queries) >= target:
                break
            queries.append({
                "query": templates[i % len(templates)].format(topic=topic),
                "ground_truth": courses,
                "query_type": "swedish_topic_search",
                "language": "sv"
            })
    return queries[:target]

def create_prerequisite_queries(prerequisite_index, target=35):
    queries = []
    templates = [
        "What are the prerequisites for {code}?",
        "Prerequisites for {code}",
        "What courses do I need before {code}?",
        "Required courses for {code}"
    ]

    courses = list(prerequisite_index.keys())
    selected = random.sample(courses, min(len(courses), (target + 3) // 4))

    for code in selected:
        for t in templates:
            if len(queries) >= target:
                break
            queries.append({
                "query": t.format(code=code),
                "ground_truth": prerequisite_index[code],
                "query_type": "prerequisite_lookup",
                "language": "en"
            })
    return queries[:target]

def create_evaluation_dataset(cleaned_courses, topics, swedish_topics):
    indexes = build_course_indexes(cleaned_courses, topics, swedish_topics)
    dataset = (
        create_topic_queries(indexes, topics, 35) +
        create_level_topic_queries(indexes, topics, ["Pre-university", "First-cycle", "Second-cycle"], 35) +
        create_credit_topic_queries(indexes, topics, 35) +
        create_specific_course_queries(cleaned_courses, 35) +
        create_swedish_topic_queries(indexes["swedish"], swedish_topics, 35) +
        create_prerequisite_queries(indexes["prerequisite"], 35)
    )
    random.shuffle(dataset)
    return dataset



topics = [
    "machine learning", "deep learning", "artificial intelligence",
    "computer vision", "natural language processing",
    "databases", "software engineering", "algorithms",
    "networks", "security", "cryptography",
    "sustainability", "robotics", "optimization",
    "signal processing", "control systems"
]

swedish_topics = [
    "maskininlärning", "djupinlärning", "artificiell intelligens",
    "datorseende", "naturlig språkbehandling",
    "databaser", "programvaruteknik", "algoritmer",
    "nätverk", "säkerhet", "kryptografi",
    "hållbarhet", "robotik", "optimering",
    "signalbehandling", "styrsystem"
]

# Generate dataset
eval_dataset = create_evaluation_dataset(cleaned, topics, swedish_topics)

# Show samples
print("="*60)
print("EVALUATION DATASET GENERATED")
print("="*60)

query_types = [
    ("Type 1", "topic_search"),
    ("Type 2", "level_topic"),
    ("Type 3", "credit_topic"),
    ("Type 4", "specific_lookup"),
    ("Type 5", "swedish_topic_search"),
    ("Type 6", "prerequisite_lookup"),
]

for label, qtype in query_types:
    matching = [q for q in eval_dataset if q['query_type'] == qtype]
    if matching:
        sample = matching[0]
        print(f"\n{label}: {sample['query']}")
        print(f"  Ground truth: {sample['ground_truth'][:5]}..." if len(sample['ground_truth']) > 5 else f"  Ground truth: {sample['ground_truth']}")
        print(f"  Total queries of this type: {len(matching)}")

print(f"\n{'='*60}")
print(f"Total queries generated: {len(eval_dataset)}")
# Save
with open("evaluation_dataset.json", "w", encoding="utf-8") as f:
    json.dump(eval_dataset, f, ensure_ascii=False, indent=2)

EVALUATION DATASET GENERATED

Type 1: Find courses about computer vision
  Ground truth: ['EEN020']
  Total queries of this type: 35

Type 2: Second-cycle courses about signal processing
  Ground truth: ['RRY057', 'SSY205', 'DAT116', 'SSY135', 'VTA151']...
  Total queries of this type: 35

Type 3: 7.5 credit courses about signal processing
  Ground truth: ['RRY057', 'SSY205', 'DAT116', 'SSY061', 'MVE290']...
  Total queries of this type: 35

Type 4: What is ACE475?
  Ground truth: ['ACE475']
  Total queries of this type: 35

Type 5: Vilka kurser handlar om hållbarhet?
  Ground truth: ['SEE050', 'TEK940', 'ACE590', 'ENM036', 'ACE246']...
  Total queries of this type: 35

Type 6: What are the prerequisites for LEU340?
  Ground truth: ['LEU076', 'LET086']
  Total queries of this type: 35

Total queries generated: 210


 ## 2: systems


**2.1: System A Base Model (No Context, no retrieval, no fine-tuning):**

In [24]:

hf_token = os.getenv("HF_TOKEN")
print(hf_token[:10] + "..." if hf_token else "Token not found")

llm_model = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    device_map="auto",
    model_kwargs={
        "torch_dtype": torch.float16
    },
    pipeline_kwargs={
        "max_new_tokens": 64,         
        "temperature": 0.3,
        "repetition_penalty": 1.2,
        "no_repeat_ngram_size": 3,
        "return_full_text": False
    }
)

print("Model Loaded")



prompt = (
    "Answer the question using ONLY the following format:\n"
    "ANSWER: <your answer>\n\n"
    "Do not add anything else.\n\n"
    "Question: In one sentence, what is the capital of France?\n"
    "ANSWER:"
)

raw_response = llm_model.invoke(prompt)
answer = raw_response.split("ANSWER:")[-1].strip().split("\n")[0]

print(f"Prompt:\n{prompt}")
print(f"\nFinal Answer:\n{answer}")


hf_wBVqymj...
Model Loaded
Prompt:
Answer the question using ONLY the following format:
ANSWER: <your answer>

Do not add anything else.

Question: In one sentence, what is the capital of France?
ANSWER:

Final Answer:
Paris.


**system A pipline:**

In [26]:

print(f"Loaded {len(eval_dataset)} evaluation queries")

# Function to extract course codes from text
def extract_course_codes(text):
    pattern = r'\b[A-Z]{3,6}\d{3}\b'
    codes = re.findall(pattern, text)
    return list(set(codes))


# Function to run System A on one query
def run_system_a_single(query_text, llm):
#     prompt = f"""Answer the question using ONLY the following format:
# ANSWER: <your answer>

# Do not add anything else.

# Question: {query_text}
# ANSWER:"""
    prompt = f"""You are answering questions about university courses.
If you are unsure, answer with "I don't know".

Answer the question using ONLY the following format:
ANSWER: <your answer>

Question: {query_text}
ANSWER:"""
    response = llm.invoke(prompt)

    predicted_codes = extract_course_codes(response)

    return {
        'response': response,
        'predicted_codes': predicted_codes
    }


# Test the extraction function
test_text = "You should take DAT450 and TDA233 courses. Maybe also FSP047."
print(f"\nTest extraction: {extract_course_codes(test_text)}")

# Test on one query
print("\n" + "="*60)
print("Testing System A on one query:")

test_query = eval_dataset[0]
print(f"\nQuery: {test_query['query']}")
print(f"Ground truth: {test_query['ground_truth']}")

result = run_system_a_single(test_query['query'], llm_model)
print(f"\nModel response: {result['response']}")
print(f"Predicted codes: {result['predicted_codes']}")

Loaded 210 evaluation queries

Test extraction: ['TDA233', 'DAT450', 'FSP047']

Testing System A on one query:

Query: What is ACE475?
Ground truth: ['ACE475']

Model response:  I don't Know.  However, it seems like a course code and could be related to Aerospace Engineering at Carleton University in Ottawa Canada. The 'ACE' prefix suggests that this may indeed be an aerospace engineering course. Without more information or context however, it's difficult to say for certain what specific topics would be
Predicted codes: []


In [27]:
print("\n" + "="*60)
print("Testing System A on different query types:")
print("="*60)
test_indices = [0, 35, 70, 105, 140, 175, 209 ]  # Different types

for idx in test_indices:
    query = eval_dataset[idx]
    print(f"\n[Type: {query['query_type']}]")
    print(f"Query: {query['query']}")
    print(f"Ground truth: {query['ground_truth']}")
    
    result = run_system_a_single(query['query'], llm_model)
    print(f"Model response: {result['response']}")
    print(f"Predicted codes: {result['predicted_codes']}")
    print("-" * 60)


Testing System A on different query types:

[Type: specific_lookup]
Query: What is ACE475?
Ground truth: ['ACE475']
Model response:  I don't Know.  ANSWEr: I dont knoW. ANSWER: I DON'T KNOW. ANSWEr: i dOn'T kNoW. anSwEr: IdonTknOw. Answer: idk. AnswEr: IDK. ansWeR: id
Predicted codes: []
------------------------------------------------------------

[Type: topic_search]
Query: Find courses about databases
Ground truth: ['DAT475', 'KBT315', 'EEN065', 'EEN060', 'TDA357', 'DAT280', 'DAT335', 'DAT076', 'ACE250', 'EEN171']
Model response:  Database Systems and Data Management. Introduction to Databases. Advanced Topics in Database Systems. Database Design and Implementation. Database Administration. Database Security. Relational Database Theory. Object-Oriented Database Systems  Database Modeling. Database Query Languages. Database Performance Optimization. Big Data Analytics. Cloud Computing for Big Data. NoSQL D
Predicted codes: []
---------------------------------------------------------

## Run System A on all eval queries:

In [28]:
print(f"Loaded {len(eval_dataset)} evaluation queries")

system_a_results = []

for i, query in enumerate(tqdm(eval_dataset, desc="Processing")):
    result = run_system_a_single(query['query'], llm_model)
    
    system_a_results.append({
        'query_id': i,
        'query': query['query'],
        'query_type': query['query_type'],
        'ground_truth': query['ground_truth'],
        'predicted_codes': result['predicted_codes'],
        'response': result['response']
    })
    
    
    if (i + 1) % 20 == 0:
        total_preds = sum(len(r['predicted_codes']) for r in system_a_results)
        print(f"\nCompleted {i+1}/{len(eval_dataset)} queries ({total_preds} codes predicted so far)")

# Save results
with open('system_a_results.json', 'w', encoding='utf-8') as f:
    json.dump(system_a_results, f, ensure_ascii=False, indent=2)

print(f"\nSystem A complete!")

print(f"Total queries processed: {len(system_a_results)}")

# Analysis
total_predictions = sum(len(r['predicted_codes']) for r in system_a_results)
queries_with_predictions = sum(1 for r in system_a_results if len(r['predicted_codes']) > 0)

print("\n" + "="*60)
print("System A Results Summary:")
print(f"Total queries: {len(system_a_results)}")
print(f"Queries with predictions: {queries_with_predictions}")
print(f"Total course codes predicted: {total_predictions}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Loaded 210 evaluation queries

Completed 20/210 queries (0 codes predicted so far)

Completed 40/210 queries (1 codes predicted so far)

Completed 60/210 queries (6 codes predicted so far)

Completed 80/210 queries (6 codes predicted so far)

Completed 100/210 queries (12 codes predicted so far)

Completed 120/210 queries (19 codes predicted so far)

Completed 140/210 queries (24 codes predicted so far)

Completed 160/210 queries (25 codes predicted so far)

Completed 180/210 queries (33 codes predicted so far)

Completed 200/210 queries (35 codes predicted so far)

System A complete!
Total queries processed: 210

System A Results Summary:
Total queries: 210
Queries with predictions: 17
Total course codes predicted: 35


In [29]:
def evaluate_system(results):
    total_correct = 0
    total_predicted = 0
    total_relevant = 0
    mrr_sum = 0
    
    for result in results:
        predicted = set(result['predicted_codes'])
        ground_truth = set(result['ground_truth'])
        
        correct = predicted & ground_truth
        total_correct += len(correct)
        total_predicted += len(predicted)
        total_relevant += len(ground_truth)
        
        if len(predicted) > 0:
            for rank, pred in enumerate(result['predicted_codes'], 1):
                if pred in ground_truth:
                    mrr_sum += 1 / rank
                    break
    
    precision = total_correct / total_predicted if total_predicted > 0 else 0
    recall = total_correct / total_relevant if total_relevant > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    mrr = mrr_sum / len(results) if len(results) > 0 else 0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'mrr': mrr
    }
#Evaluate system and save metrics, overall + by type
def evaluate_and_save(results, system_name):
   
    
    overall_metrics = evaluate_system(results)
    
    print("\n" + "="*60)
    print(f"{system_name.upper()}: OVERALL METRICS")
    print("="*60)
    print(f"Precision: {overall_metrics['precision']:.2%}")
    print(f"Recall:    {overall_metrics['recall']:.2%}")
    print(f"F1-Score:  {overall_metrics['f1']:.2%}")
    print(f"MRR:       {overall_metrics['mrr']:.4f}")
    print("="*60)
    

    
    
    print(f"{system_name.upper()}: METRICS BY QUERY TYPE")
  
    
    by_type = {}
    for result in results:
        qtype = result['query_type']
        if qtype not in by_type:
            by_type[qtype] = []
        by_type[qtype].append(result)
    
    type_metrics = {}
    for qtype in sorted(by_type.keys()):
        metrics = evaluate_system(by_type[qtype])
        type_metrics[qtype] = metrics
        
        print(f"\n{qtype} ({len(by_type[qtype])} queries):")
        print(f"  Precision: {metrics['precision']:.2%}")
        print(f"  Recall:    {metrics['recall']:.2%}")
        print(f"  F1-Score:  {metrics['f1']:.2%}")
        print(f"  MRR:       {metrics['mrr']:.4f}")
    
    # Save detailed metrics
    detailed_metrics = {
        'overall': overall_metrics,
        'by_type': type_metrics
    }
    
    with open(f'{system_name}_detailed_metrics.json', 'w') as f:
        json.dump(detailed_metrics, f, indent=2)

    
    return overall_metrics, type_metrics

system_a_overall, system_a_by_type = evaluate_and_save(system_a_results, 'system_a')


SYSTEM_A: OVERALL METRICS
Precision: 11.43%
Recall:    0.26%
F1-Score:  0.51%
MRR:       0.0190
SYSTEM_A: METRICS BY QUERY TYPE

credit_topic (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

level_topic (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

prerequisite_lookup (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

specific_lookup (35 queries):
  Precision: 66.67%
  Recall:    11.43%
  F1-Score:  19.51%
  MRR:       0.1143

swedish_topic_search (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

topic_search (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000


## System B:

## System B1 RETRIEVAL-ONLY

In [40]:
# Load cleaned courses
with open("cleaned_courses.json", "r", encoding="utf-8") as f:
    cleaned_courses = json.load(f)
print(f"Loaded {len(cleaned_courses)} courses")

# Load evaluation dataset
with open("evaluation_dataset.json", "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)
print(f"Loaded {len(eval_dataset)} queries")

# Create vector store
timestamp = int(time.time())
persist_dir = f"./S{timestamp}"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

documents = []
for course in cleaned_courses:
    doc = Document(
        page_content=course["full_text"],
        metadata={
            "course_code": course["courseCode"],
            "credits": course["creditNumeric"],
            "level": course["eduLevelName"],
            "department": course["deptBenamning"],
        }
    )
    documents.append(doc)

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="chalmers_courses_b1",
    persist_directory=persist_dir
)
#System B1 retrieval function (NO LLM)
def run_system_b1(query_text, vectorstore, k=5):
    retrieved_docs = vectorstore.similarity_search(query_text, k=k)
    retrieved_codes = [doc.metadata["course_code"] for doc in retrieved_docs]

    return {
        "predicted_codes": retrieved_codes,
        "retrieved_codes": retrieved_codes
    }

test_query = eval_dataset[0]

print("Query:", test_query["query"])
print("Ground truth:", test_query["ground_truth"])

result = run_system_b1(test_query["query"], vectorstore, k=5)

print("Retrieved codes:", result["retrieved_codes"])

gt = set(test_query["ground_truth"])
pred = set(result["retrieved_codes"])




Loaded 1388 courses
Loaded 210 queries
Query: What is ACE475?
Ground truth: ['ACE475']
Retrieved codes: ['TMV200', 'TRA415', 'EDA264', 'EEN171', 'PPU010']


In [39]:
# test the diffrent types
test_indices = [0, 35, 70, 105, 140, 175, 209]

for idx in test_indices:
    query = eval_dataset[idx]

    print(f"\n[Type: {query['query_type']}]")
    print(f"Query: {query['query']}")
    print(f"Ground truth: {query['ground_truth']}")

    result = run_system_b1(query["query"], vectorstore, k=5)

    print(f"Retrieved codes (Top-5): {result['retrieved_codes']}")

    # overlap check
    gt = set(query["ground_truth"])
    pred = set(result["retrieved_codes"])
    overlap = gt & pred

    print(f"Correctly retrieved: {list(overlap)}")
    


[Type: specific_lookup]
Query: What is ACE475?
Ground truth: ['ACE475']
Retrieved codes (Top-5): ['TMV200', 'TRA415', 'EDA264', 'EEN171', 'PPU010']
Correctly retrieved: []

[Type: topic_search]
Query: Find courses about databases
Ground truth: ['DAT475', 'KBT315', 'EEN065', 'EEN060', 'TDA357', 'DAT280', 'DAT335', 'DAT076', 'ACE250', 'EEN171']
Retrieved codes (Top-5): ['TDA357', 'DAT335', 'DAT475', 'EEN060', 'DAT050']
Correctly retrieved: ['DAT335', 'EEN060', 'DAT475', 'TDA357']

[Type: level_topic]
Query: Second-cycle courses about machine learning
Ground truth: ['RRY057', 'TDA233', 'KBT315', 'EEN100', 'IMS135', 'DAT635', 'DAT341', 'DAT550', 'ENM097', 'DAT465', 'DAT625', 'DAT441', 'TRA375', 'DAT480', 'TIF345', 'MTF271', 'SSY210', 'TRA460', 'TRA440', 'SSY316', 'SSY340', 'DAT450', 'DAT410', 'TRA385', 'TIF360', 'TIF285', 'TRA300', 'TRA445']
Retrieved codes (Top-5): ['DAT441', 'DAT530', 'TRA385', 'DAT341', 'TIN093']
Correctly retrieved: ['DAT441', 'TRA385', 'DAT341']

[Type: specific_look

In [42]:
# run all the evalset
system_b1_results = []

for i, query in enumerate(tqdm(eval_dataset, desc="System B1")):
    result = run_system_b1(query["query"], vectorstore, k=5)

    system_b1_results.append({
        "query_id": i,
        "query": query["query"],
        "query_type": query["query_type"],
        "ground_truth": query["ground_truth"],
        "predicted_codes": result["predicted_codes"],
        "retrieved_codes": result["retrieved_codes"],
    })

# Save results
with open("system_b1_results.json", "w", encoding="utf-8") as f:
    json.dump(system_b1_results, f, indent=2)

In [43]:
system_b1_overall, system_b1_by_type = evaluate_and_save(system_b1_results, 'system_b1')


SYSTEM_B1: OVERALL METRICS
Precision: 17.52%
Recall:    12.03%
F1-Score:  14.26%
MRR:       0.3152
SYSTEM_B1: METRICS BY QUERY TYPE

credit_topic (35 queries):
  Precision: 29.14%
  Recall:    14.45%
  F1-Score:  19.32%
  MRR:       0.5167

level_topic (35 queries):
  Precision: 25.71%
  Recall:    15.90%
  F1-Score:  19.65%
  MRR:       0.4976

prerequisite_lookup (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

specific_lookup (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

swedish_topic_search (35 queries):
  Precision: 0.57%
  Recall:    0.46%
  F1-Score:  0.51%
  MRR:       0.0286

topic_search (35 queries):
  Precision: 49.71%
  Recall:    15.21%
  F1-Score:  23.29%
  MRR:       0.8486


## System B2: RAG = Retrieval (B1) + LLM selection

In [44]:

def run_system_b2(query_text, vectorstore, llm, k=5):
    # Step 1: Retrieve candidates 
    retrieved_docs = vectorstore.similarity_search(query_text, k=k)
    retrieved_codes = [doc.metadata["course_code"] for doc in retrieved_docs]

    
    context = []
    for doc in retrieved_docs:
        context.append(
            f"{doc.metadata['course_code']}: {doc.page_content[:500]}"
        )

    context_text = "\n\n".join(context)

    # Step 2: LLM selection prompt
    prompt = f"""
You are selecting relevant university courses.

Context (retrieved courses):
{context_text}

Question:
{query_text}

Rules:
- ONLY output course codes from the context
- Output as a comma-separated list
- If none match, output: NONE
- Do NOT explain

Answer:
"""

    response = llm.invoke(prompt)

    # Step 3: Extract and filter codes
    predicted_codes = extract_course_codes(response)
    predicted_codes = [c for c in predicted_codes if c in retrieved_codes]

    return {
        "response": response,
        "predicted_codes": predicted_codes,
        "retrieved_codes": retrieved_codes
    }


In [45]:

system_b2_results = []

for i, query in enumerate(tqdm(eval_dataset, desc="System B2")):
    result = run_system_b2(query["query"], vectorstore, llm_model, k=5)

    system_b2_results.append({
        "query_id": i,
        "query": query["query"],
        "query_type": query["query_type"],
        "ground_truth": query["ground_truth"],
        "predicted_codes": result["predicted_codes"],
        "retrieved_codes": result["retrieved_codes"],
        "response": result["response"]
    })

with open("system_b2_results.json", "w") as f:
    json.dump(system_b2_results, f, indent=2)
system_b2_overall, system_b2_by_type = evaluate_and_save(system_b2_results, "system_b2")



SYSTEM_B2: OVERALL METRICS
Precision: 31.91%
Recall:    10.72%
F1-Score:  16.05%
MRR:       0.2607
SYSTEM_B2: METRICS BY QUERY TYPE

credit_topic (35 queries):
  Precision: 34.68%
  Recall:    12.18%
  F1-Score:  18.03%
  MRR:       0.4452

level_topic (35 queries):
  Precision: 38.83%
  Recall:    14.13%
  F1-Score:  20.73%
  MRR:       0.4571

prerequisite_lookup (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

specific_lookup (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

swedish_topic_search (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

topic_search (35 queries):
  Precision: 58.27%
  Recall:    14.16%
  F1-Score:  22.78%
  MRR:       0.6619


## System c

In [47]:

# Load cleaned courses
with open("cleaned_courses.json", "r", encoding="utf-8") as f:
    cleaned_courses = json.load(f)

print(f"Loaded courses: {len(cleaned_courses)}")

# Show one sample
sample = cleaned_courses[0]
print("\nSample course:")
print("Code:", sample["courseCode"])
print("Name:", sample["name"])
print("Credits:", sample["creditNumeric"])
print("Level:", sample["eduLevelName"])
print("Has full_text:", "full_text" in sample)
print("Full_text length:", len(sample["full_text"]))


Loaded courses: 1388

Sample course:
Code: FSP047
Name: English for engineers
Credits: 7.5
Level: First-cycle
Has full_text: True
Full_text length: 2349


In [48]:
# mode: - 'code' : CODE - 'list' : CODE (Course name) - 'full' : CODE + short structured description
def format_course_response(course, mode="list"):
   
    if mode == "code":
        return course["courseCode"]

    if mode == "list":
        return f'{course["courseCode"]} ({course["name"]})'

    if mode == "full":
        return (
            f'{course["courseCode"]} is "{course["name"]}", '
            f'a {course["creditNumeric"]} HP {course["eduLevelName"]} course '
            f'at Chalmers University.'
        )

    raise ValueError("Unknown mode")
print(format_course_response(cleaned_courses[0], "code"))
print(format_course_response(cleaned_courses[0], "list"))
print(format_course_response(cleaned_courses[0], "full"))

FSP047
FSP047 (English for engineers)
FSP047 is "English for engineers", a 7.5 HP First-cycle course at Chalmers University.


**generate tranings set**

In [49]:

def generate_specific_lookups(courses, n=500):
    examples = []

    templates = [
        "What is {code}?",
        "Tell me about {code}",
        "Describe {code}",
        "Give me information about {code}",
        "What does {code} cover?"
    ]

    sampled = random.sample(courses, min(len(courses), n))

    for course in sampled:
        instruction = random.choice(templates).format(
            code=course["courseCode"]
        )

        output = format_course_response(course, mode="full")

        examples.append({
            "instruction": instruction,
            "input": "",
            "output": output
        })

    return examples


specific_examples = generate_specific_lookups(cleaned_courses, n=500)

print(f"Generated {len(specific_examples)} specific lookup examples")

for i in range(3):
    print("\n--- Example ---")
    print("Instruction:", specific_examples[i]["instruction"])
    print("Output:", specific_examples[i]["output"])


Generated 500 specific lookup examples

--- Example ---
Instruction: Describe TRA295
Output: TRA295 is "Technology, politics, and society", a 15.0 HP Second-cycle course at Chalmers University.

--- Example ---
Instruction: What does TIF120 cover?
Output: TIF120 is "Surface and nanophysics", a 7.5 HP Second-cycle course at Chalmers University.

--- Example ---
Instruction: Describe MVE570
Output: MVE570 is "Linear algebra and differential equations", a 7.5 HP First-cycle course at Chalmers University.


In [50]:

def generate_topic_searches(courses, topics, n=300):
    examples = []

    templates = [
        "What courses teach {topic}?",
        "Find courses about {topic}",
        "I want to learn about {topic}",
        "Show me courses related to {topic}"
    ]

    # Build topic index
    topic_index = defaultdict(list)
    for course in courses:
        text = course["full_text"].lower()
        for topic in topics:
            if topic in text:
                topic_index[topic].append(course)

    count = 0
    attempts = 0

    while count < n and attempts < n * 3:
        attempts += 1
        topic = random.choice(topics)

        if topic not in topic_index or len(topic_index[topic]) == 0:
            continue

        selected = random.sample(
            topic_index[topic],
            min(len(topic_index[topic]), random.randint(4, 7))
        )

        instruction = random.choice(templates).format(topic=topic)
        output = ", ".join(
            format_course_response(c, mode="list") for c in selected
        )

        examples.append({
            "instruction": instruction,
            "input": "",
            "output": output
        })
        count += 1

    return examples


topic_examples = generate_topic_searches(
    cleaned_courses,
    topics=[
        "machine learning", "deep learning", "artificial intelligence",
        "databases", "computer vision", "robotics",
        "signal processing", "control systems"
    ],
    n=300
)

print(f"Generated {len(topic_examples)} topic search examples")


for i in range(3):
    print("\n--- Example ---")
    print("Instruction:", topic_examples[i]["instruction"])
    print("Output:", topic_examples[i]["output"])


Generated 300 topic search examples

--- Example ---
Instruction: Show me courses related to machine learning
Output: SSY340 (Deep machine learning), TIF345 (Advanced simulation and machine learning), KBT315 (Advanced analytical chemistry), FFR135 (Artificial neural networks), DAT635 (Machine learning in healthcare), IMS135 (Machine learning and data-driven modelling in mechanics), TIF285 (Learning from data)

--- Example ---
Instruction: Show me courses related to computer vision
Output: TIF160 (Humanoid robotics), EEN020 (Computer vision), SSY098 (Image analysis), SSY340 (Deep machine learning)

--- Example ---
Instruction: I want to learn about artificial intelligence
Output: FSC011 (Policymaking for climate action and circular economy: Project course), TRA300 (Digitalization in sports: From physics to innovation), DAT565 (Introduction to data science and AI), IMS065 (Data science in product realization), MMS131 (Introduction to artificial intelligence), TRA320 (Music engineering: A

In [51]:

def generate_level_topic_searches(courses, topics, n=200):
    examples = []

    levels = ["First-cycle", "Second-cycle", "Pre-university"]

    templates = [
        "{level} courses about {topic}",
        "Find {level} courses on {topic}",
        "Show me {level} {topic} courses"
    ]

    # Build index: level → topic → courses
    index = defaultdict(lambda: defaultdict(list))
    for course in courses:
        level = course["eduLevelName"]
        text = course["full_text"].lower()
        for topic in topics:
            if topic in text:
                index[level][topic].append(course)

    count = 0
    attempts = 0

    while count < n and attempts < n * 3:
        attempts += 1
        level = random.choice(levels)
        topic = random.choice(topics)

        if topic not in index[level] or len(index[level][topic]) == 0:
            continue

        selected = random.sample(
            index[level][topic],
            min(len(index[level][topic]), random.randint(3, 6))
        )

        instruction = random.choice(templates).format(
            level=level,
            topic=topic
        )

        output = ", ".join(
            format_course_response(c, mode="list") for c in selected
        )

        examples.append({
            "instruction": instruction,
            "input": "",
            "output": output
        })
        count += 1

    return examples


level_topic_examples = generate_level_topic_searches(
    cleaned_courses,
    topics=[
        "machine learning", "deep learning", "artificial intelligence",
        "databases", "computer vision", "robotics",
        "signal processing", "control systems"
    ],
    n=200
)

print(f"Generated {len(level_topic_examples)} level+topic examples")

# Show samples
for i in range(3):
    print("\n--- Example ---")
    print("Instruction:", level_topic_examples[i]["instruction"])
    print("Output:", level_topic_examples[i]["output"])



Generated 200 level+topic examples

--- Example ---
Instruction: First-cycle courses about signal processing
Output: SSY061 (Mechatronics), EEN225 (Introduction to biomedical engineering: Digital), EEN071 (Introduction to biomedical engineering), MVE101 (Transforms and differential equations), LET271 (Electrical measurements: Systems and methods)

--- Example ---
Instruction: First-cycle courses about deep learning
Output: EEN175 (Introduction to machine learning), MMS285 (Digitalization and AI for future shipping: Fundamentals and applications), MMS131 (Introduction to artificial intelligence)

--- Example ---
Instruction: Show me Second-cycle deep learning courses
Output: BOM075 (Drinking water engineering), TRA415 (Industrial immersion in AI and cybersecurity), TRA235 (Data-driven product realization)


In [52]:
def generate_credit_topic_searches(courses, topics, n=150):
    examples = []

    credits = [7.5, 15.0, 30.0]

    templates = [
        "{credit} credit courses about {topic}",
        "Find {credit} HP courses on {topic}",
        "Show me {credit} credit {topic} courses"
    ]

    # Build index: credit → topic → courses
    index = defaultdict(lambda: defaultdict(list))
    for course in courses:
        credit = course["creditNumeric"]
        text = course["full_text"].lower()
        for topic in topics:
            if topic in text:
                index[credit][topic].append(course)

    count = 0
    attempts = 0

    while count < n and attempts < n * 3:
        attempts += 1
        credit = random.choice(credits)
        topic = random.choice(topics)

        if topic not in index[credit] or len(index[credit][topic]) == 0:
            continue

        selected = random.sample(
            index[credit][topic],
            min(len(index[credit][topic]), random.randint(3, 5))
        )

        instruction = random.choice(templates).format(
            credit=credit,
            topic=topic
        )

        output = ", ".join(
            format_course_response(c, mode="list") for c in selected
        )

        examples.append({
            "instruction": instruction,
            "input": "",
            "output": output
        })
        count += 1

    return examples


credit_topic_examples = generate_credit_topic_searches(
    cleaned_courses,
    topics=[
        "machine learning", "deep learning", "artificial intelligence",
        "databases", "computer vision", "robotics"
    ],
    n=150
)

print(f"Generated {len(credit_topic_examples)} credit+topic examples")

for i in range(3):
    print("\n--- Example ---")
    print("Instruction:", credit_topic_examples[i]["instruction"])
    print("Output:", credit_topic_examples[i]["output"])


Generated 150 credit+topic examples

--- Example ---
Instruction: Find 15.0 HP courses on machine learning
Output: TRA375 (Digitalization in sports: From physics to innovation)

--- Example ---
Instruction: 15.0 credit courses about machine learning
Output: TRA375 (Digitalization in sports: From physics to innovation)

--- Example ---
Instruction: 7.5 credit courses about artificial intelligence
Output: TRA320 (Music engineering: Awareness of sound), TRA420 (Modeling climate futures: Science, economics, ethics and policy), MMS131 (Introduction to artificial intelligence), IMS065 (Data science in product realization)


In [53]:

def generate_swedish_queries(courses, swedish_topics, n=50):
    examples = []

    templates = [
        "Vilka kurser handlar om {topic}?",
        "Hitta kurser om {topic}",
        "Visa mig kurser om {topic}",
        "Kurser som täcker {topic}"
    ]

    # Build Swedish index
    index = defaultdict(list)
    for course in courses:
        name_alt = course.get("nameAlt", "").lower()
        for topic in swedish_topics:
            if topic in name_alt:
                index[topic].append(course)

    count = 0
    attempts = 0

    while count < n and attempts < n * 3:
        attempts += 1
        topic = random.choice(swedish_topics)

        if topic not in index or len(index[topic]) == 0:
            continue

        selected = random.sample(
            index[topic],
            min(len(index[topic]), random.randint(3, 5))
        )

        instruction = random.choice(templates).format(topic=topic)
        output = ", ".join(
            format_course_response(c, mode="code") for c in selected
        )

        examples.append({
            "instruction": instruction,
            "input": "",
            "output": output
        })
        count += 1

    return examples


swedish_examples = generate_swedish_queries(
    cleaned_courses,
    swedish_topics=[
        "maskininlärning", "djupinlärning", "artificiell intelligens",
        "databaser", "robotik", "styrsystem"
    ],
    n=50
)

print(f"Generated {len(swedish_examples)} Swedish examples")

# Show samples
for i in range(3):
    print("\n--- Example ---")
    print("Instruction:", swedish_examples[i]["instruction"])
    print("Output:", swedish_examples[i]["output"])


Generated 50 Swedish examples

--- Example ---
Instruction: Kurser som täcker robotik
Output: TRA455

--- Example ---
Instruction: Hitta kurser om databaser
Output: TDA357, DAT475

--- Example ---
Instruction: Vilka kurser handlar om robotik?
Output: TRA455


In [54]:

# Combine all training examples together
training_data = (
    specific_examples
    + topic_examples
    + level_topic_examples
    + credit_topic_examples
    + swedish_examples
)

# Shuffle 
random.shuffle(training_data)

print(f"TOTAL TRAINING EXAMPLES: {len(training_data)}")

# Show a few random samples
for i in range(3):
    ex = random.choice(training_data)
    print("\n--- Sample ---")
    print("Instruction:", ex["instruction"])
    print("Output:", ex["output"])

# Save
with open("training_data.json", "w", encoding="utf-8") as f:
    json.dump(training_data, f, ensure_ascii=False, indent=2)



TOTAL TRAINING EXAMPLES: 1200

--- Sample ---
Instruction: Give me information about TMS165
Output: TMS165 is "Stochastic calculus", a 7.5 HP Second-cycle course at Chalmers University.

--- Sample ---
Instruction: What courses teach artificial intelligence?
Output: MMS131 (Introduction to artificial intelligence), TRA320 (Music engineering: Awareness of sound), EEN095 (Artificial intelligence and autonomous systems), TRA385 (Machine learning and AI through artistic innovation), FSC005 (Policymaking for climate action and circular economy), TRA300 (Digitalization in sports: From physics to innovation)

--- Sample ---
Instruction: Find courses about machine learning
Output: DAT441 (Advanced topics in machine learning), SSY210 (Information theory, advanced level), FFR105 (Stochastic optimization algorithms), EEN100 (Statistics and machine learning in high dimensions)

Saved training_data.json


In [55]:


print("Loading base model for System C...")

model_name = "meta-llama/Llama-3.1-8B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)


model.config.use_cache = False
print(f"Model device: {model.device}")


Loading base model for System C...
Model device: cuda:1


In [56]:

print("Preparing model for LoRA training")

# Prepare  model
model = prepare_model_for_kbit_training(model)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Attach LoRA adapters
model = get_peft_model(model, lora_config)
# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable %:          {100 * trainable_params / total_params:.2f}%")


Preparing model for LoRA training
Trainable parameters: 6,815,744
Total parameters:     4,547,416,064
Trainable %:          0.15%


In [57]:
# Format each example as a LLaMA-3 instruction-style conversation

print("Formatting training data for LLaMA-3")
 

def format_instruction(sample):
   
    instruction = sample["instruction"]
    output = sample["output"]

    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are a helpful university course advisor for Chalmers University of Technology."
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{instruction}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{output}"
        "<|eot_id|>"
    )

    return prompt

# Load training data
with open("training_data.json", "r", encoding="utf-8") as f:
    training_data = json.load(f)

# Format all examples
formatted_texts = [format_instruction(sample) for sample in training_data]

# Create dataset
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"Formatted {len(dataset)} training examples")

# Show one example
print("\n--- Sample formatted example ---")
print(dataset[0]["text"][:800])


Formatting training data for LLaMA-3
Formatted 1200 training examples

--- Sample formatted example ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful university course advisor for Chalmers University of Technology.<|eot_id|><|start_header_id|>user<|end_header_id|>
Tell me about TEK656<|eot_id|><|start_header_id|>assistant<|end_header_id|>
TEK656 is "Creating technology-based ventures", a 7.5 HP Second-cycle course at Chalmers University.<|eot_id|>


In [58]:
print("Tokenizing dataset")

def tokenize_function(batch):
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)

# one example
print("\nTokenized sample")
print("Input IDs length:", len(tokenized_dataset[0]["input_ids"]))
print("Labels length:", len(tokenized_dataset[0]["labels"]))


Tokenizing dataset


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1200
})

--- Tokenized sample ---
Input IDs length: 512
Labels length: 512


In [59]:

print("Setting up training configuration")

training_args = TrainingArguments(
    output_dir="./chalmers_lora_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,      
    gradient_accumulation_steps=8,  
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=50,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)


Setting up training configuration...
Trainer initialized.


In [60]:

print("Starting system C training (LoRA)")
trainer.train()


Starting system C training (LoRA)


Step,Training Loss
10,4.947300
20,4.127300
30,3.153200
40,2.104600
50,1.691800
60,1.512600
70,1.587500
80,1.251800
90,1.047600
100,1.060400


TrainOutput(global_step=450, training_loss=0.8904102145300972, metrics={'train_runtime': 1919.4193, 'train_samples_per_second': 1.876, 'train_steps_per_second': 0.234, 'total_flos': 8.30738396086272e+16, 'train_loss': 0.8904102145300972, 'epoch': 3.0})

In [61]:

adapter_path = "./chalmers_lora_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

('./chalmers_lora_adapter/tokenizer_config.json',
 './chalmers_lora_adapter/special_tokens_map.json',
 './chalmers_lora_adapter/tokenizer.json')

In [62]:

print("Loading System C (Fine-tuned LoRA model)")


model_name = "meta-llama/Llama-3.1-8B-Instruct"
adapter_path = "./chalmers_lora_adapter"

# Load tokenizer 
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit quantization config 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print(f"Model device: {model.device}")


Loading System C (Fine-tuned LoRA model)
Model device: cuda:1


In [75]:
def is_single_course_query(query):
    return bool(re.search(r"\b[A-Z]{3,6}\d{3}\b", query))
def run_system_c_single(query_text, model, tokenizer):
    prompt = f"""You are a helpful university course advisor for Chalmers University.

IMPORTANT OUTPUT RULES:

- If the question is about ONE specific course code (e.g. "What is ACE475?"):
  → Output EXACTLY ONE line in this format:
    CODE: <course_code> | NAME: <course_name>
  → STOP immediately after that line.

- If the question asks for MULTIPLE courses:
  → Output a comma-separated list of course codes ONLY.

Question: {query_text}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        single = is_single_course_query(query_text)
        outputs = model.generate(
        **inputs,
        max_new_tokens=32 if single else 64,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=4,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in response:
        response = response.split("Answer:")[-1].strip()

    response = response.split("\n")[0].strip()

    predicted_codes = extract_course_codes(response)
    if single and len(predicted_codes) > 0:
        predicted_codes = [predicted_codes[0]]
    return {
        "response": response,
        "predicted_codes": predicted_codes,
    }

test_query = eval_dataset[0]

print("Query:", test_query["query"])
print("Ground truth:", test_query["ground_truth"])

result = run_system_c_single(
    test_query["query"],
    model,
    tokenizer
)

print("\nModel response:")
print(result["response"])

print("Predicted course codes:")
print(result["predicted_codes"])


Query: What is ACE475?
Ground truth: ['ACE475']

Model response:
ACE475, Building physics and building technology, Introduction to architecture [15.0 HP] - First-cycle course
Predicted course codes:
['ACE475']


In [76]:
#Testing System C on different query types
test_indices = [0, 35, 70, 105, 140, 175, 209]  

for idx in test_indices:
    query = eval_dataset[idx]

    print(f"\n[Type: {query['query_type']}]")
    print(f"Query: {query['query']}")
    print(f"Ground truth: {query['ground_truth']}")

    result = run_system_c_single(
        query["query"],
        model,
        tokenizer
    )

    print(f"Model response: {result['response']}")
    print(f"Predicted codes: {result['predicted_codes']}")

    # overlap check
    gt = set(query["ground_truth"])
    pred = set(result["predicted_codes"])
    overlap = gt & pred

    print(f"Correct predictions: {list(overlap)}")




[Type: specific_lookup]
Query: What is ACE475?
Ground truth: ['ACE475']
Model response: ACE475, Building physics and building technology, Introduction to architecture [15.0 HP] - First-cycle course
Predicted codes: ['ACE475']
Correct predictions: ['ACE475']

[Type: topic_search]
Query: Find courses about databases
Ground truth: ['DAT475', 'KBT315', 'EEN065', 'EEN060', 'TDA357', 'DAT280', 'DAT335', 'DAT076', 'ACE250', 'EEN171']
Model response: DAT060, TDA233, PPU125, TEK240, KBT315, TRA385, SSY205, MPP083, EEN065, BOM075, LMT108, FSC011, DDB131, RRY057, VTA132, SJO072, DAT
Predicted codes: ['BOM075', 'DAT060', 'SSY205', 'MPP083', 'RRY057', 'TDA233', 'TEK240', 'TRA385', 'PPU125', 'SJO072', 'LMT108', 'FSC011', 'DDB131', 'VTA132', 'KBT315', 'EEN065']
Correct predictions: ['KBT315', 'EEN065']

[Type: level_topic]
Query: Second-cycle courses about machine learning
Ground truth: ['RRY057', 'TDA233', 'KBT315', 'EEN100', 'IMS135', 'DAT635', 'DAT341', 'DAT550', 'ENM097', 'DAT465', 'DAT625', 'DA

In [77]:
#Running System C on all queries

system_c_results = []

for i, query in enumerate(tqdm(eval_dataset, desc="System C")):
    result = run_system_c_single(query["query"], model, tokenizer)

    system_c_results.append({
        "query_id": i,
        "query": query["query"],
        "query_type": query["query_type"],
        "ground_truth": query["ground_truth"],
        "predicted_codes": result["predicted_codes"],
        "response": result["response"],
    })

with open("system_c_results.json", "w", encoding="utf-8") as f:
    json.dump(system_c_results, f, indent=2)

system_c_overall, system_c_by_type = evaluate_and_save(system_c_results,"system_c")



SYSTEM_C: OVERALL METRICS
Precision: 8.69%
Recall:    8.37%
F1-Score:  8.52%
MRR:       0.1809
SYSTEM_C: METRICS BY QUERY TYPE

credit_topic (35 queries):
  Precision: 7.36%
  Recall:    7.65%
  F1-Score:  7.50%
  MRR:       0.2599

level_topic (35 queries):
  Precision: 5.87%
  Recall:    10.60%
  F1-Score:  7.56%
  MRR:       0.1528

prerequisite_lookup (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

specific_lookup (35 queries):
  Precision: 31.25%
  Recall:    28.57%
  F1-Score:  29.85%
  MRR:       0.2857

swedish_topic_search (35 queries):
  Precision: 7.04%
  Recall:    6.94%
  F1-Score:  6.99%
  MRR:       0.1476

topic_search (35 queries):
  Precision: 14.60%
  Recall:    8.04%
  F1-Score:  10.37%
  MRR:       0.2394


In [ ]:
## System d RAG + Fine-tuned model

In [79]:

with open("cleaned_courses.json", "r", encoding="utf-8") as f:
    cleaned_courses = json.load(f)
print(f"Loaded {len(cleaned_courses)} courses")

with open("evaluation_dataset.json", "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)
print(f"Loaded {len(eval_dataset)} eval queries")

timestamp = int(time.time())
persist_dir = f"./chroma_db_d_{timestamp}"
print("Persist dir:", persist_dir)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

documents = []
for c in cleaned_courses:
    documents.append(
        Document(
            page_content=c["full_text"],
            metadata={
                "course_code": c["courseCode"],
                "course_name": c.get("name", ""),
                "credits": c.get("creditNumeric", None),
                "level": c.get("eduLevelName", ""),
                "department": c.get("deptBenamning", ""),
            },
        )
    )

print(f"Prepared {len(documents)} documents")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="chalmers_courses_system_d",
    persist_directory=persist_dir,
)




Loaded 1388 courses
Loaded 210 eval queries
Persist dir: ./chroma_db_d_1767660080
Prepared 1388 documents


In [82]:


def extract_query_course_codes(query):
    return re.findall(r"\b[A-Z]{3,6}\d{3}\b", query)


def run_system_d_single(query_text, vectorstore, model, tokenizer, k=5):

    # Step 1: Retrieve
    retrieved_docs = vectorstore.similarity_search(query_text, k=k)
    retrieved_codes = [doc.metadata["course_code"] for doc in retrieved_docs]

    # Step 2: Detect specific lookup
    query_codes = extract_query_course_codes(query_text)
    is_specific = len(query_codes) == 1


    if is_specific and query_codes[0] not in retrieved_codes:
        return {
            "response": "No relevant courses found.",
            "predicted_codes": [],
            "retrieved_codes": retrieved_codes
        }

    
    # Step 3: Build context
    context = []
    for doc in retrieved_docs:
        context.append(f"{doc.metadata['course_code']}: {doc.page_content[:500]}")

    context_text = "\n\n".join(context)


    prompt = f"""You are a helpful university course advisor for Chalmers University.

IMPORTANT RULES:
- You may ONLY mention course codes present in the context.
- If none match, say: No relevant courses found.
- If the question is about ONE course, answer only about that course.
- If the question asks for MULTIPLE courses, output comma-separated course codes only.

Context:
{context_text}

Question: {query_text}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in response:
        response = response.split("Answer:")[-1].strip()

    response = response.split("\n")[0].strip()

    predicted_codes = extract_course_codes(response)
    predicted_codes = [c for c in predicted_codes if c in retrieved_codes]

    return {
        "response": response,
        "predicted_codes": predicted_codes,
        "retrieved_codes": retrieved_codes
    }


In [83]:

# Test on ONE query
test_query = eval_dataset[0]
result = run_system_d_single(
    test_query["query"],
    vectorstore,
    model,
    tokenizer,
    k=5
)
print(result)


{'response': 'No relevant courses found.', 'predicted_codes': [], 'retrieved_codes': ['TMV200', 'TRA415', 'EDA264', 'EEN171', 'PPU010']}


In [84]:
#"Testing System D on all the different query types
test_indices = [0, 35, 70, 105, 140, 175, 209]

for idx in test_indices:
    query = eval_dataset[idx]

    print(f"\n[Type: {query['query_type']}]")
    print(f"Query: {query['query']}")
    print(f"Ground truth: {query['ground_truth']}")

    result = run_system_d_single(
        query["query"],
        vectorstore,
        model,
        tokenizer,
        k=5
    )

    print(f"Retrieved codes: {result['retrieved_codes']}")
    print(f"Model response: {result['response']}")
    print(f"Predicted codes: {result['predicted_codes']}")

    # overlap check
    gt = set(query["ground_truth"])
    pred = set(result["predicted_codes"])
    overlap = gt & pred

    print(f"Correct predictions: {list(overlap)}")



[Type: specific_lookup]
Query: What is ACE475?
Ground truth: ['ACE475']
Retrieved codes: ['TMV200', 'TRA415', 'EDA264', 'EEN171', 'PPU010']
Model response: No relevant courses found.
Predicted codes: []
Correct predictions: []

[Type: topic_search]
Query: Find courses about databases
Ground truth: ['DAT475', 'KBT315', 'EEN065', 'EEN060', 'TDA357', 'DAT280', 'DAT335', 'DAT076', 'ACE250', 'EEN171']
Retrieved codes: ['TDA357', 'DAT335', 'DAT475', 'EEN060', 'DAT050']
Model response: DAT475, TMA131, SSY251, TRA385, PPU125, ERE103, MPP083, VTA132, KBT315, SJO598, BOM075, LMT108, DBN230, RTO436, TEK240, DAT460, DAT540
Predicted codes: ['DAT475']
Correct predictions: ['DAT475']

[Type: level_topic]
Query: Second-cycle courses about machine learning
Ground truth: ['RRY057', 'TDA233', 'KBT315', 'EEN100', 'IMS135', 'DAT635', 'DAT341', 'DAT550', 'ENM097', 'DAT465', 'DAT625', 'DAT441', 'TRA375', 'DAT480', 'TIF345', 'MTF271', 'SSY210', 'TRA460', 'TRA440', 'SSY316', 'SSY340', 'DAT450', 'DAT410', 'TR

In [85]:
# Running System D on all queries
system_d_results = []

for i, query in enumerate(tqdm(eval_dataset, desc="System D")):
    result = run_system_d_single(query["query"],vectorstore,model,tokenizer,k=5)

    system_d_results.append({
        "query_id": i,
        "query": query["query"],
        "query_type": query["query_type"],
        "ground_truth": query["ground_truth"],
        "predicted_codes": result["predicted_codes"],
        "retrieved_codes": result["retrieved_codes"],
        "response": result["response"],
    })
    if (i + 1) % 20 == 0:
        total_preds = sum(len(r['predicted_codes']) for r in system_d_results)
        print(f"\nCompleted {i+1}/{len(eval_dataset)} queries ({total_preds} codes predicted so far)")

with open("system_d_results.json", "w", encoding="utf-8") as f:
    json.dump(system_d_results, f, indent=2)

system_d_overall, system_d_by_type = evaluate_and_save(system_d_results,"system_d")



Completed 20/210 queries (6 codes predicted so far)

Completed 40/210 queries (10 codes predicted so far)

Completed 60/210 queries (12 codes predicted so far)

Completed 80/210 queries (22 codes predicted so far)

Completed 100/210 queries (24 codes predicted so far)

Completed 120/210 queries (29 codes predicted so far)

Completed 140/210 queries (30 codes predicted so far)

Completed 160/210 queries (32 codes predicted so far)

Completed 180/210 queries (33 codes predicted so far)

Completed 200/210 queries (34 codes predicted so far)

SYSTEM_D: OVERALL METRICS
Precision: 55.88%
Recall:    1.24%
F1-Score:  2.43%
MRR:       0.0667
SYSTEM_D: METRICS BY QUERY TYPE

credit_topic (35 queries):
  Precision: 42.86%
  Recall:    0.85%
  F1-Score:  1.67%
  MRR:       0.0857

level_topic (35 queries):
  Precision: 60.00%
  Recall:    1.06%
  F1-Score:  2.08%
  MRR:       0.0857

prerequisite_lookup (35 queries):
  Precision: 0.00%
  Recall:    0.00%
  F1-Score:  0.00%
  MRR:       0.0000

sp

In [86]:
free_questions = [
    "I want to study machine learning at Chalmers, what courses should I look at?",
    "What courses cover deep learning?",
    "Are there any courses about databases and data management?",
    "Which 7.5 credit courses are related to AI?",
    "Second-cycle courses in computer vision?",
    "What is DAT450?",
    "Tell me about ACE475.",
    "What do I need before DAT595?",
    "Which courses require linear algebra?",
    "I want something related to robotics and control, any suggestions?",
]


def _safe_run(system_name, fn, *args, **kwargs):
    try:
        out = fn(*args, **kwargs)
        return {"ok": True, "output": out, "error": None}
    except Exception as e:
        return {"ok": False, "output": None, "error": f"{type(e).__name__}: {e}"}


def run_A(q):
    out = run_system_a_single(q, llm_model)
    return {"response": out.get("response", ""), "predicted_codes": out.get("predicted_codes", [])}

def run_B1(q, k=5):
    out = run_system_b1(q, vectorstore, k=k)
    return {"response": None, "predicted_codes": out.get("predicted_codes", out.get("retrieved_codes", []))}

def run_B2(q, k=5):
    out = run_system_b2(q, vectorstore, llm_model, k=k)
    return {"response": out.get("response", ""), "predicted_codes": out.get("predicted_codes", [])}

def run_C(q):
    out = run_system_c_single(q, model, tokenizer)
    return {"response": out.get("response", ""), "predicted_codes": out.get("predicted_codes", [])}

def run_D(q, k=5):
    out = run_system_d_single(q, vectorstore, model, tokenizer, k=k)
    return {"response": out.get("response", ""), "predicted_codes": out.get("predicted_codes", [])}

results = []


for i, q in enumerate(free_questions, 1):
    print("\n" + "=" * 90)
    print(f"Q{i}: {q}")
    print("=" * 90)

    rA  = _safe_run("A",  run_A,  q)
    rB1 = _safe_run("B1", run_B1, q, 5)
    rB2 = _safe_run("B2", run_B2, q, 5)
    rC  = _safe_run("C",  run_C,  q)
    rD  = _safe_run("D",  run_D,  q, 5)

    def _print_block(name, r):
        print(f"\n{name}:")
        if not r["ok"]:
            print("  ERROR:", r["error"])
            return

        out = r["output"]
        codes = out.get("predicted_codes", [])
        resp = out.get("response", None)

        print("  Codes:", codes if codes else "[]")
        if resp is not None:
            text = resp.strip()
            if len(text) > 500:
                text = text[:500] + " ..."
            print("  Response:", text)

    _print_block("System A",  rA)
    _print_block("System B1", rB1)
    _print_block("System B2", rB2)
    _print_block("System C",  rC)
    _print_block("System D",  rD)

    results.append({
        "id": i,
        "question": q,
        "A":  rA,
        "B1": rB1,
        "B2": rB2,
        "C":  rC,
        "D":  rD,
    })

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = f"free_query_tests_{ts}.json"




Q1: I want to study machine learning at Chalmers, what courses should I look at?


/data/users/omarala/a1env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/users/omarala/a1env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



System A:
  Codes: []
  Response: Machine Learning (ML), Artificial Intelligence and Data Mining.  Also consider Computer Vision and Deep Learning if interested in those areas.  Additionally, Linear Algebra, Calculus, Probability Theory and Statistics for a solid mathematical background.  For programming skills Python is recommended as it's widely used in ML.   And finally, also

System B1:
  Codes: ['DAT441', 'DAT530', 'DAT341', 'EEN095', 'TIN093']

System B2:
  Codes: ['DAT441', 'DAT341', 'TIN093', 'DAT530', 'EEN095']
  Response: DAT441,DAT530,DAT341,EEN095,TIN093 DAT341,DAT441,Dat530 

Corrected Answer:
DAT341,Dat441,DAt530 
(removed duplicates) 
 Corrected Answer after re-checking rules again.
DAT441, DAT530, DAT341, EEN095,

System C:
  Codes: ['DAT565', 'TDA233']
  Response: DAT565, TDA233

System D:
  Codes: []
  Response: DAT440, TIF345, MMS131, ESS013, SSY316, PPU125, DAT635, DAT236, DAT340, DAT625, DAT235, DAT295, DAT230, LMT108, BOM075, TRA385, DAT285, KBT315, R

Q2: What c

/data/users/omarala/a1env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/users/omarala/a1env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



System A:
  Codes: []
  Response: Machine Learning and Artificial Intelligence. Computer Vision and Natural Language Processing also often include topics on Deep Learning.  Additionally, there may be a dedicated course or specialisation in Deep Learning itself.  Some universities offer an entire degree program focused solely on AI/ML/DL.   However, this can vary widely depending on the

System B1:
  Codes: ['SSY340', 'FFR135', 'TIF360', 'DAT441', 'TRA385']

System B2:
  Codes: ['SSY340', 'FFR135', 'TIF360']
  Response: SSY320, FFR135, TIF360 

Explanation:

* SSY340 mentions "deep machine learning" 
* FFR130 was not found but FFR125 was not present either however there's another course named FFR136 wasn't available so I looked at FFR133 nope then i saw

System C:
  Codes: ['DAT565', 'TDA233']
  Response: DAT565, TDA233

System D:
  Codes: ['FFR135', 'TRA385']
  Response: DAT341, SSY316, TDA233, FFR135, PPU125, MMS131, TRA385, SSO236, EEN175, LMT108, FFZ065, BOM075, DAT615, RRY057, KBT

...


System A:
  Codes: []
  Response: Completion of 3 credits in Data Science or Statistics. Also completion of MATH 220 and STAT 301 is recommended but not required. Additionally, it's assumed that students have a basic understanding of Python programming language.  It’s also highly suggested to take this course after taking either CS 210 (Data Structures)

System B1:
  Codes: ['EDA234', 'MMT016', 'PPU196', 'DAT096', 'CIU177']

System B2:
  Codes: ['DAT096']
  Response: NONE
DAT095 does not exist in the retrieved data; however, DAT096 exists but it's a second cycle course. Therefore, there isn't any prerequisite information available about DAT595 since it doesn't appear in the provided text. However, based on the pattern of other courses listed, if we were looking for prerequisites

System C:
  Codes: ['TIF160']
  Response: TIF160, ESS013

System D:
  Codes: []
  Response: No relevant courses found.

Q9: Which courses require linear algebra?


/data/users/omarala/a1env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/users/omarala/a1env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



System A:
  Codes: []
  Response: Calculus III, Differential Equations. Linear Algebra, Vector Calculus. Multivariable Calculus and Differential Geometry also often do.  Also some physics and engineering classes may use it as well.  In general any class that involves vector spaces or matrices will probably need to have a good understanding of linear algebra.

System B1:
  Codes: ['MVE675', 'TMV206', 'TMV216', 'MVE670', 'MVE610']

System B2:
  Codes: ['TMV206', 'TMV216']
  Response: MVE615,MVE620,TMV106,TMV116,TMV126,TMV136,TMV146,TMV156,TMV166,TMV176,TMV186,TMV196,TMV206,TMV216,TMV226,TMV236,TMV246,TMV256,TMV266,TMV276,TMV286,T

System C:
  Codes: ['TDA233', 'DAT335']
  Response: DAT335, TDA233

System D:
  Codes: []
  Response: TMV106, MMS131, DAT335, EEN225, SSY305, LBT432, RRY057, PPU125, VTA132, BOM235, FSC101, KMB056, SJO598, TEK240, ENM060, PPQ071, DAT

Q10: I want something related to robotics and control, any suggestions?


/data/users/omarala/a1env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/users/omarala/a1env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



System A:
  Codes: []
  Response: Yes. You might be interested in Control Systems or Mechatronics Engineering. Both of these fields involve designing systems that can interact with their environment through sensors and actuators. Additionally, there's also Robotics which is a more specific field focused on creating robots capable of performing tasks autonomously.  Which one sounds most appealing?

System B1:
  Codes: ['DAT295', 'SSY236', 'EEN095', 'TME290', 'SSY263']

System B2:
  Codes: ['DAT295', 'EEN095', 'TME290']
  Response: DAT295,TME290,EEN095,SSY363,SSSY263 
( Note: SSYS236 was not selected because it had "robotic" but no "control")  -> corrected answer without extra character.
 DAT295,TM290,Een095,ssy263  -> Corrected answer removing

System C:
  Codes: ['SSY205', 'MCC125', 'LMT108']
  Response: MCC125, LMT108, SSY205

System D:
  Codes: []
  Response: DAT565, TIF160, SSY052, ERE103, RRY057, LMT125, PPU127, MMS196, TRA385, TME146, SJO598, BOM235, DAT335, DAT550, DRT105, MCC13

In [ ]:
HHHH